In [1]:
# =============================================================================
#  IBM Backend Selection & Transpilation Analysis
#  Probabilistic Yoshida–Kitaev Decoder (Implementation A)
#  Paper: "Closed Timelike Curves Enabled by Decoder Circuits on a Quantum Processor"
# =============================================================================

# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector, partial_trace, state_fidelity, DensityMatrix
from qiskit_ibm_runtime import QiskitRuntimeService

# =============================================================================
#  STEP 0 — IBM Quantum Authentication
# =============================================================================
# Paste your IBM Quantum API token below (get it from https://quantum.ibm.com)
IBM_TOKEN = "QfkScNfX4bVJ5lXm0082x7F7J6vya3SF5LJZFNOYXqqO"

print("=" * 65)
print("  IBM Quantum Backend Selection — Probabilistic YK Decoder")
print("=" * 65)

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_TOKEN)
print("\n[✓] Authenticated with IBM Quantum.\n")

# =============================================================================
#  STEP 1 — Build the Probabilistic Decoder Circuit (Implementation A)
#           Lines 44-136 of your probabilistic_decoder.py, cleaned up
# =============================================================================

def build_probabilistic_decoder(theta=2.5349076035276403,
                                varphi=2.0022404587009195):
    """
    Builds the probabilistic Yoshida-Kitaev decoder circuit
    (post-selected, Lloyd-type CTC emulation — Implementation A).

    Registers: C, E, R, G, M, A, Y  (7 qubits total)
    Classical: crR, crG, crC        (3 bits)
    """
    C = QuantumRegister(1, 'C')   # CTC qubit
    E = QuantumRegister(1, 'E')   # Early radiation
    R = QuantumRegister(1, 'R')   # Recent radiation
    G = QuantumRegister(1, 'G')   # Grover / flag qubit
    M = QuantumRegister(1, 'M')   # Message qubit
    A = QuantumRegister(1, 'A')   # Ancilla
    Y = QuantumRegister(1, 'Y')   # Output

    crR = ClassicalRegister(1, 'crR')
    crG = ClassicalRegister(1, 'crG')
    crC = ClassicalRegister(1, 'crC')

    qc = QuantumCircuit(C, E, R, G, M, A, Y, crR, crG, crC)

    # --- Prepare message on M ---
    qc.u(theta, varphi, 0.0, M)

    # --- SWAP message into CTC register ---
    qc.swap(C, M)
    qc.barrier()

    # --- Bell pairs: (E,M), (R,G), (A,Y) ---
    qc.h(E);  qc.cx(E, M)
    qc.h(R);  qc.cx(R, G)
    qc.h(A);  qc.cx(A, Y)
    qc.barrier()

    # --- Scrambling unitary U on (C, E, R) ---
    qc.cz(C, R);  qc.cz(E, R);  qc.cz(C, E)
    qc.h(C);  qc.h(E);  qc.h(R)
    qc.cz(C, R);  qc.cz(C, E);  qc.cz(E, R)
    qc.barrier()

    # --- Probabilistic decoder: unitary conjugate U† on (A, M, G) ---
    qc.cz(A, G);  qc.cz(M, A);  qc.cz(G, M)
    qc.h(A);  qc.h(M);  qc.h(G)
    qc.cz(A, G);  qc.cz(G, M);  qc.cz(M, A)
    qc.barrier()

    # --- Bell projection on (R, G) — post-select on |Φ⁺⟩ ---
    qc.cx(R, G)
    qc.h(R)
    qc.measure(R, crR)
    qc.measure(G, crG)
    qc.barrier()

    # --- SWAP output Y back into CTC register C ---
    qc.swap(C, Y)
    qc.u(theta, varphi, 0.0, C).inverse()
    qc.measure(C, crC)

    return qc


qc = build_probabilistic_decoder()
print(f"[✓] Circuit built: {qc.num_qubits} qubits, {qc.num_clbits} classical bits")
print(f"    Pre-transpile depth       : {qc.depth()}")
print(f"    Pre-transpile 2q gate ops : "
      f"{sum(1 for _, qargs, _ in qc.data if len(qargs) == 2)}\n")

# =============================================================================
#  STEP 2 — List available open-tier backends with >= 7 qubits
# =============================================================================

print("Fetching available backends (operational, >= 7 qubits) ...")
all_backends = service.backends(operational=True, simulator=False, min_num_qubits=7)

if not all_backends:
    raise RuntimeError("No operational backends found with >= 7 qubits. "
                       "Check your IBM account or try later.")

print(f"  Found {len(all_backends)} candidate backend(s):\n")
for b in all_backends:
    cfg = b.configuration()
    print(f"    • {b.name:30s}  {cfg.n_qubits} qubits")

# =============================================================================
#  STEP 3 — Transpile against each backend and collect metrics
# =============================================================================

print("\nTranspiling circuit against each backend (optimization_level=3) ...")
print("-" * 65)

results = []

for backend in all_backends:
    name = backend.name
    try:
        t_circ = transpile(
            qc,
            backend=backend,
            optimization_level=3,   # heaviest optimization to minimize 2q gates
            seed_transpiler=42
        )

        depth       = t_circ.depth()
        two_q_count = sum(1 for _, qargs, _ in t_circ.data if len(qargs) == 2)

        results.append({
            "backend"    : name,
            "n_qubits"   : backend.configuration().n_qubits,
            "depth"      : depth,
            "two_q_gates": two_q_count,
            "t_circ"     : t_circ,
            "backend_obj": backend
        })

        print(f"  {name:30s}  depth={depth:4d}  2q-gates={two_q_count:4d}")

    except Exception as e:
        print(f"  {name:30s}  FAILED: {e}")

if not results:
    raise RuntimeError("Transpilation failed for all backends.")

# =============================================================================
#  STEP 4 — Pick the best backend (minimize two-qubit gate count)
# =============================================================================

best = min(results, key=lambda x: (x["two_q_gates"], x["depth"]))
print(f"\n[★] Best backend: {best['backend']}  "
      f"(depth={best['depth']}, 2q-gates={best['two_q_gates']})\n")

# =============================================================================
#  STEP 5 — Pull calibration snapshot for the winning backend
# =============================================================================

print("=" * 65)
print(f"  Calibration Snapshot — {best['backend']}")
print("=" * 65)

backend_obj = best["backend_obj"]
props       = backend_obj.properties()
t_circ      = best["t_circ"]

# ── Identify which physical qubits are actually used ──────────────────────
used_qubits = sorted(set(
    t_circ.find_bit(q).index
    for _, qargs, _ in t_circ.data
    for q in qargs
))

print(f"\nPhysical qubits used by transpiled circuit: {used_qubits}\n")

# ── Readout (measurement) error rates ─────────────────────────────────────
print("Readout error rates (per qubit):")
print(f"  {'Qubit':>6}  {'Readout Error':>14}")
print(f"  {'-----':>6}  {'-------------':>14}")

readout_errors = {}
for q in used_qubits:
    try:
        err = props.readout_error(q)
    except Exception:
        err = float('nan')
    readout_errors[q] = err
    print(f"  q[{q:02d}]    {err:.6f}")

# ── CX / ECR / CZ gate error rates for used two-qubit pairs ───────────────
print("\nTwo-qubit gate error rates (edges used in transpiled circuit):")
print(f"  {'Edge':>10}  {'Gate':>6}  {'Error':>10}")
print(f"  {'----':>10}  {'----':>6}  {'-----':>10}")

two_q_errors = {}
for instr, qargs, _ in t_circ.data:
    if len(qargs) == 2:
        q0 = t_circ.find_bit(qargs[0]).index
        q1 = t_circ.find_bit(qargs[1]).index
        edge = (min(q0, q1), max(q0, q1))
        if edge not in two_q_errors:
            gate_name = instr.name
            try:
                err = props.gate_error(gate_name, [q0, q1])
            except Exception:
                try:
                    err = props.gate_error(gate_name, [q1, q0])
                except Exception:
                    err = float('nan')
            two_q_errors[edge] = (gate_name, err)
            print(f"  q{edge[0]}–q{edge[1]:>3}     {gate_name:>6}  {err:.6f}")

# ── Summary table ──────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  SUMMARY — Paper-ready numbers (Implementation A, IBM backend)")
print("=" * 65)
print(f"\n  Backend selected     : {best['backend']}")
print(f"  Number of qubits     : {best['n_qubits']}")
print(f"  Physical qubits used : {used_qubits}")
print(f"  Transpiled depth     : {best['depth']}")
print(f"  Two-qubit gate count : {best['two_q_gates']}")

avg_readout = np.nanmean(list(readout_errors.values()))
avg_two_q   = np.nanmean([v for _, v in two_q_errors.values()])
print(f"  Avg readout error    : {avg_readout:.4f}  ({avg_readout*100:.2f}%)")
print(f"  Avg 2q-gate error    : {avg_two_q:.4f}  ({avg_two_q*100:.2f}%)")

print("\n  Per-qubit readout errors:")
for q, e in readout_errors.items():
    print(f"    q[{q:02d}]: {e:.4f}")

print("\n  Per-edge two-qubit gate errors:")
for (q0, q1), (gate, e) in two_q_errors.items():
    print(f"    q{q0}–q{q1} ({gate}): {e:.4f}")

print("\n[✓] Done. Use the numbers above for Section IV A of the paper.\n")

# =============================================================================
#  STEP 6 — Save the transpiled circuit diagram (optional)
# =============================================================================
try:
    t_circ.draw(output='mpl', filename='transpiled_decoder_ibm.png',
                fold=40, style='iqp')
    print("[✓] Transpiled circuit saved to 'transpiled_decoder_ibm.png'")
except Exception as e:
    print(f"[!] Could not save circuit diagram: {e}")

qiskit_runtime_service._discover_account:WARNING:2026-03-19 12:11:41,698: Loading account with the given token. A saved account will not be used.


  IBM Quantum Backend Selection — Probabilistic YK Decoder


qiskit_runtime_service.__init__:WARNING:2026-03-19 12:11:45,111: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: CTCs. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-03-19 12:11:45,117: Loading instance: CTCs, plan: open



[✓] Authenticated with IBM Quantum.

[✓] Circuit built: 7 qubits, 3 classical bits
    Pre-transpile depth       : 24
    Pre-transpile 2q gate ops : 18

Fetching available backends (operational, >= 7 qubits) ...
  Found 4 candidate backend(s):

    • ibm_fez                         156 qubits
    • ibm_torino                      133 qubits
    • ibm_marrakesh                   156 qubits
    • ibm_kingston                    156 qubits

Transpiling circuit against each backend (optimization_level=3) ...
-----------------------------------------------------------------
  ibm_fez                         depth=  87  2q-gates=  35
  ibm_torino                      depth=  87  2q-gates=  32
  ibm_marrakesh                   depth=  87  2q-gates=  35
  ibm_kingston                    depth=  87  2q-gates=  35

[★] Best backend: ibm_torino  (depth=87, 2q-gates=32)

  Calibration Snapshot — ibm_torino

Physical qubits used by transpiled circuit: [45, 46, 55, 65, 66, 67, 68]

Readout error r